# Expansão Monte Carlo do estado para RL

Este notebook produz trajetórias sintéticas a partir do `rl_state.parquet`. Ele é um protótipo de pesquisa: amplia o conjunto de treino e permite experimentar perturbações de regime, mas não substitui dados reais nem validação fora da amostra.

## Método

O método padrão é um *moving block bootstrap*. Em vez de sortear linhas isoladas, ele sorteia blocos temporais contíguos da matriz de estado. Cada linha preserva a relação transversal entre as 28 features; cada bloco preserva dependência local no tempo. As trajetórias são identificadas por `trajectory_id` e `step`, portanto não devem ser confundidas com uma única série histórica real.

## Hipóteses de regime

Os parâmetros abaixo permitem testes controlados: `location_shift_std` desloca cada feature em unidades de seu desvio padrão; `dispersion_multiplier` amplia ou contrai desvios em torno da média; e `noise_scale` acrescenta ruído gaussiano multivariado proporcional à covariância observada. O padrão `baseline` não aplica alteração de regime. Esses cenários são hipóteses sintéticas, não previsões de mercado.

In [1]:
import hashlib
import json
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

## Configuração

Altere somente esta célula para criar experiências diferentes. A semente deve ser preservada no manifesto para que uma execução possa ser reproduzida. `N_TRAJECTORIES=100` cria 100 trajetórias de 205 passos cada, totalizando 20.500 estados sintéticos.

In [2]:
INPUT_PATH = Path("C:/projects/Libellula/data/processed/rl_state/rl_state.parquet")
OUTPUT_DIR = Path("C:/projects/Libellula/data/processed/rl_state/monte_carlo_datasets")
OUTPUT_PATH = OUTPUT_DIR / "rl_state_monte_carlo.parquet"
METADATA_PATH = OUTPUT_DIR / "rl_state_monte_carlo.metadata.json"

SEED = 42
N_TRAJECTORIES = 100
BLOCK_SIZE = 20

REGIME = {
    "name": "baseline",
    "location_shift_std": 0.0,
    "dispersion_multiplier": 1.0,
    "noise_scale": 0.0,
}

## Carregamento e verificações

O estado de entrada deve conter apenas features numéricas, índice temporal único e nenhum valor ausente. Estas verificações impedem que o simulador esconda problemas nos insumos reais.

In [3]:
state = pd.read_parquet(INPUT_PATH).sort_index()

assert isinstance(state.index, pd.DatetimeIndex)
assert state.index.is_monotonic_increasing
assert not state.index.has_duplicates
assert state.select_dtypes(exclude=np.number).empty
assert not state.isna().any().any()
assert BLOCK_SIZE > 0
assert N_TRAJECTORIES > 0
assert REGIME["dispersion_multiplier"] >= 0
assert REGIME["noise_scale"] >= 0

FEATURE_COLUMNS = list(state.columns)
print(f"Estado real: {state.shape[0]} linhas x {state.shape[1]} features")
print(f"Período: {state.index.min().date()} a {state.index.max().date()}")

Estado real: 205 linhas x 28 features
Período: 2024-07-18 a 2025-04-30


## Gerador de trajetórias

A função sorteia posições iniciais e copia blocos com retorno circular ao começo da série quando necessário. Em seguida, aplica a transformação de regime em torno da média observada. Ruído, quando ativado, é multivariado para respeitar a estrutura de covariância em vez de perturbar cada feature independentemente.

In [4]:
def sha256(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()


def sample_positions(length: int, block_size: int, rng: np.random.Generator) -> np.ndarray:
    positions = []
    while len(positions) < length:
        start = int(rng.integers(0, length))
        block = (start + np.arange(block_size)) % length
        positions.extend(block.tolist())
    return np.asarray(positions[:length])


def apply_regime(values: np.ndarray, source: pd.DataFrame, rng: np.random.Generator) -> np.ndarray:
    mean = source.mean().to_numpy()
    std = source.std(ddof=0).replace(0, 1.0).to_numpy()
    adjusted = mean + REGIME["dispersion_multiplier"] * (values - mean)
    adjusted = adjusted + REGIME["location_shift_std"] * std

    if REGIME["noise_scale"] > 0:
        covariance = source.cov().to_numpy()
        covariance = covariance + np.eye(covariance.shape[0]) * 1e-12
        noise = rng.multivariate_normal(
            mean=np.zeros(values.shape[1]),
            cov=covariance * REGIME["noise_scale"] ** 2,
            size=len(values),
        )
        adjusted = adjusted + noise

    return adjusted

## Simulação

Cada trajetória possui o mesmo comprimento do estado real. `source_timestamp` registra a observação histórica de onde cada estado foi amostrado; ele é uma coluna de auditoria e não uma feature para o futuro agente RL. As features que o agente poderá receber estão listadas em `FEATURE_COLUMNS`.

In [5]:
rng = np.random.default_rng(SEED)
values = state.to_numpy(dtype=np.float64)
trajectories = []

for trajectory_id in range(N_TRAJECTORIES):
    positions = sample_positions(len(state), BLOCK_SIZE, rng)
    simulated_values = apply_regime(values[positions], state, rng)
    trajectory = pd.DataFrame(simulated_values, columns=FEATURE_COLUMNS)
    trajectory.insert(0, "source_timestamp", state.index[positions])
    trajectory.insert(0, "step", np.arange(len(state), dtype=np.int32))
    trajectory.insert(0, "trajectory_id", trajectory_id)
    trajectories.append(trajectory)

synthetic_state = pd.concat(trajectories, ignore_index=True)
synthetic_state[FEATURE_COLUMNS] = synthetic_state[FEATURE_COLUMNS].astype(np.float32)

assert len(synthetic_state) == len(state) * N_TRAJECTORIES
assert not synthetic_state[FEATURE_COLUMNS].isna().any().any()
synthetic_state.head()

,trajectory_id,step,source_timestamp,epf_upper,epf_lower,epf_width,epf_asymmetry,quantile_lower,q25,q50,...,pdfd_03,pdf_skew,pdf_kurtosis,evt_shape,evt_scale,evt_var,evt_cvar,prediction,pred_return,confidence
0,0,0,2024-08-13,1.628522,0.569878,1.058644,0.0,-0.006687,-0.003644,-0.000722,...,0.0,0.158336,1.661852,1.298252,0.000379,0.120183,-0.381974,0.000109,0.000109,0.000109
1,0,1,2024-08-14,1.626446,0.575954,1.050491,0.0,-0.006687,-0.002621,-0.001023,...,0.0,0.157646,1.659677,0.459155,0.000658,0.016019,0.026095,0.000161,0.000161,0.000161
2,0,2,2024-08-15,1.620861,0.573339,1.047523,0.0,-0.006797,-0.002465,-0.000186,...,0.0,0.171788,1.693816,0.459155,0.000658,0.016019,0.026095,0.000094,0.000094,0.000094
3,0,3,2024-08-16,1.629504,0.576096,1.053407,0.0,-0.006797,-0.003436,-0.000714,...,0.0,0.172807,1.678786,0.459155,0.000658,0.016019,0.026095,0.000195,0.000195,0.000195
4,0,4,2024-08-19,1.635154,0.581846,1.053308,0.0,-0.006508,-0.002708,-0.001643,...,0.0,0.178641,1.699700,0.459155,0.000658,0.016019,0.026095,0.000276,0.000276,0.000276


## Salvamento e rastreabilidade

O manifesto registra o arquivo de origem, seu hash, a semente, o método, os parâmetros de regime e o hash da saída. Ele é necessário para distinguir experimentos sintéticos e permitir sua reprodução.

In [6]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
synthetic_state.to_parquet(OUTPUT_PATH, index=False)

metadata = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "method": "moving_block_bootstrap_with_optional_regime_transform",
    "input": {
        "path": str(INPUT_PATH),
        "sha256": sha256(INPUT_PATH),
        "rows": len(state),
        "features": FEATURE_COLUMNS,
    },
    "simulation": {
        "seed": SEED,
        "trajectories": N_TRAJECTORIES,
        "steps_per_trajectory": len(state),
        "block_size": BLOCK_SIZE,
        "regime": REGIME,
    },
    "output": {
        "path": str(OUTPUT_PATH),
        "sha256": sha256(OUTPUT_PATH),
        "rows": len(synthetic_state),
        "columns": list(synthetic_state.columns),
    },
}
METADATA_PATH.write_text(json.dumps(metadata, indent=2), encoding="utf-8")
print(OUTPUT_PATH)
print(METADATA_PATH)

C:\projects\Libellula\data\processed\rl_state\monte_carlo_datasets\rl_state_monte_carlo.parquet
C:\projects\Libellula\data\processed\rl_state\monte_carlo_datasets\rl_state_monte_carlo.metadata.json


## Leitura correta da saída

O arquivo contém dados artificiais. Para treino de RL, `trajectory_id` e `step` definem a ordem de cada episódio; `source_timestamp` serve exclusivamente à auditoria. O modelo RL futuro deverá usar apenas `FEATURE_COLUMNS` como observação, até que sejam definidos ações, recompensa e regras de transição na etapa seguinte.